# Student Health Risk Prediction

## Data Preprocessing and Experimental Preparation

**Module:** CIS6005 Computational Intelligence  
**Competition:** Predicting Student Health Risk  
**Competition Series:** Kaggle Playground Series – Season 6, Episode 7

### Notebook Objective

This notebook prepares the student health-risk dataset for four classification approaches:

1. Logistic Regression
2. Random Forest
3. Support Vector Machine
4. Multi-Layer Perceptron Neural Network

The preprocessing strategy is informed by the exploratory data analysis conducted in the previous notebook. The main issues identified were:

- severe target-class imbalance;
- missing values in both numerical and categorical predictors;
- mixed numerical and categorical data types;
- differences in numerical feature scales;
- a limited proportion of potential outliers;
- no severe multicollinearity among numerical predictors.

To reduce data leakage, preprocessing transformations are defined using scikit-learn pipelines and are intended to be fitted only on the training partition.

In [27]:
import os
import sys
import json
import warnings
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn import __version__ as sklearn_version
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42

print("Python Version:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Pandas Version:", pd.__version__)
print("NumPy Version:", np.__version__)
print("Scikit-learn Version:", sklearn_version)
print("Random State:", RANDOM_STATE)

Python Version: 3.14.0
Platform: macOS-26.5.1-arm64-arm-64bit-Mach-O
Pandas Version: 3.0.3
NumPy Version: 2.4.6
Scikit-learn Version: 1.8.0
Random State: 42


## 1. Load the Competition Datasets

The training, test, and sample-submission files are loaded from the raw-data directory.

The training dataset contains the target variable `health_condition`, whereas the test dataset contains only predictor variables and the identifier required for Kaggle submission.

In [28]:
TRAIN_PATH = "../data/raw/train.csv"
TEST_PATH = "../data/raw/test.csv"
SUBMISSION_PATH = "../data/raw/sample_submission.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SUBMISSION_PATH)

print("Training Shape:", train_df.shape)
print("Test Shape:", test_df.shape)
print("Sample Submission Shape:", sample_submission.shape)

Training Shape: (690088, 15)
Test Shape: (295753, 14)
Sample Submission Shape: (295753, 2)


In [29]:
TARGET_COLUMN = "health_condition"
ID_COLUMN = "id"

print("Target Column:", TARGET_COLUMN)
print("ID Column:", ID_COLUMN)

print("\nTarget exists in training data:",
      TARGET_COLUMN in train_df.columns)

print("Target exists in test data:",
      TARGET_COLUMN in test_df.columns)

print("\nDuplicate training rows:",
      train_df.duplicated().sum())

print("Duplicate test rows:",
      test_df.duplicated().sum())

print("\nTarget missing values:",
      train_df[TARGET_COLUMN].isna().sum())

print("\nTarget classes:")
print(train_df[TARGET_COLUMN].value_counts())

Target Column: health_condition
ID Column: id

Target exists in training data: True
Target exists in test data: False

Duplicate training rows: 0
Duplicate test rows: 0

Target missing values: 0

Target classes:
health_condition
at-risk      592561
unhealthy     57724
fit           39803
Name: count, dtype: int64


## 2. Separate Predictors, Target, and Identifier

The `id` attribute is retained separately because it is required for the final Kaggle submission but is not treated as a predictive health feature.

The target variable `health_condition` is separated from the predictor matrix before train-validation splitting.

In [30]:
train_ids = train_df[ID_COLUMN].copy()
test_ids = test_df[ID_COLUMN].copy()

X = train_df.drop(columns=[TARGET_COLUMN, ID_COLUMN]).copy()
y = train_df[TARGET_COLUMN].copy()

X_test = test_df.drop(columns=[ID_COLUMN]).copy()

print("Training Predictor Shape:", X.shape)
print("Target Shape:", y.shape)
print("Competition Test Predictor Shape:", X_test.shape)

print("\nTraining and test feature columns match:",
      list(X.columns) == list(X_test.columns))

print("\nPredictor Columns:")
print(X.columns.tolist())

Training Predictor Shape: (690088, 13)
Target Shape: (690088,)
Competition Test Predictor Shape: (295753, 13)

Training and test feature columns match: True

Predictor Columns:
['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake', 'diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']


## 3. Define Numerical and Categorical Features

The dataset contains seven continuous numerical predictors and six categorical predictors.

Separate feature groups are required because different preprocessing operations are appropriate for each data type.

In [31]:
numerical_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

print("\nNumber of Numerical Features:",
      len(numerical_features))

print("Number of Categorical Features:",
      len(categorical_features))

print("\nTotal Model Features:",
      len(numerical_features) + len(categorical_features))

Numerical Features:
['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']

Categorical Features:
['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']

Number of Numerical Features: 7
Number of Categorical Features: 6

Total Model Features: 13


In [32]:
schema_check = pd.DataFrame({
    "Feature": X.columns,
    "Train_Dtype": [str(X[col].dtype) for col in X.columns],
    "Test_Dtype": [str(X_test[col].dtype) for col in X.columns],
    "Train_Missing": [X[col].isna().sum() for col in X.columns],
    "Test_Missing": [X_test[col].isna().sum() for col in X.columns]
})

schema_check["Dtype_Match"] = (
    schema_check["Train_Dtype"]
    == schema_check["Test_Dtype"]
)

schema_check

,Feature,Train_Dtype,Test_Dtype,Train_Missing,Test_Missing,Dtype_Match
0,sleep_duration,float64,float64,75999,32571,True
1,heart_rate,float64,float64,7833,3357,True
2,bmi,float64,float64,13898,5956,True
3,calorie_expenditure,float64,float64,52853,22652,True
4,step_count,float64,float64,13916,5964,True
5,exercise_duration,float64,float64,6901,2958,True
6,water_intake,float64,float64,43477,18633,True
7,diet_type,str,str,6901,2958,True
8,stress_level,str,str,82811,35490,True
9,sleep_quality,str,str,58331,24999,True


## 4. Stratified Train-Validation Split

Exploratory analysis identified severe class imbalance:

- `at-risk`: 85.87%
- `unhealthy`: 8.36%
- `fit`: 5.77%

The majority-to-minority ratio was approximately 14.89:1.

A stratified split is therefore used so that the class proportions are preserved in both the training and validation partitions. A fixed random state improves reproducibility.

In [33]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train Shape:", X_train.shape)
print("X_valid Shape:", X_valid.shape)
print("y_train Shape:", y_train.shape)
print("y_valid Shape:", y_valid.shape)

X_train Shape: (552070, 13)
X_valid Shape: (138018, 13)
y_train Shape: (552070,)
y_valid Shape: (138018,)


In [34]:
def class_distribution(series):
    counts = series.value_counts()
    percentages = (
        series.value_counts(normalize=True) * 100
    ).round(2)

    return pd.DataFrame({
        "Count": counts,
        "Percentage": percentages
    })

print("Full Dataset Distribution")
display(class_distribution(y))

print("\nTraining Distribution")
display(class_distribution(y_train))

print("\nValidation Distribution")
display(class_distribution(y_valid))

Full Dataset Distribution


,Count,Percentage
health_condition,,
at-risk,592561,85.87
unhealthy,57724,8.36
fit,39803,5.77



Training Distribution


,Count,Percentage
health_condition,,
at-risk,474049,85.87
unhealthy,46179,8.36
fit,31842,5.77



Validation Distribution


,Count,Percentage
health_condition,,
at-risk,118512,85.87
unhealthy,11545,8.36
fit,7961,5.77


In [35]:
assert len(X_train) == len(y_train)
assert len(X_valid) == len(y_valid)

assert X_train.index.intersection(X_valid.index).empty

assert set(X_train.columns) == set(X_valid.columns)
assert set(X_train.columns) == set(X_test.columns)

print("Split integrity checks passed.")
print("No row overlap between training and validation partitions.")
print("Feature schemas are consistent.")

Split integrity checks passed.
No row overlap between training and validation partitions.
Feature schemas are consistent.


## 5. Target Encoding

The target labels are converted into numerical form for compatibility with the selected classification algorithms.

A `LabelEncoder` is fitted using the training target partition. The mapping is stored so that numerical model predictions can later be converted back to the original Kaggle class labels.

In [36]:
target_encoder = LabelEncoder()

y_train_encoded = target_encoder.fit_transform(y_train)
y_valid_encoded = target_encoder.transform(y_valid)

target_mapping = {
    class_name: int(encoded_value)
    for encoded_value, class_name
    in enumerate(target_encoder.classes_)
}

inverse_target_mapping = {
    value: key
    for key, value in target_mapping.items()
}

print("Target Classes:")
print(target_encoder.classes_)

print("\nTarget Mapping:")
print(target_mapping)

print("\nEncoded Training Target Shape:",
      y_train_encoded.shape)

print("Encoded Validation Target Shape:",
      y_valid_encoded.shape)

Target Classes:
['at-risk' 'fit' 'unhealthy']

Target Mapping:
{'at-risk': 0, 'fit': 1, 'unhealthy': 2}

Encoded Training Target Shape: (552070,)
Encoded Validation Target Shape: (138018,)


In [37]:
encoding_verification = pd.DataFrame({
    "Original_Label": y_train.iloc[:10].values,
    "Encoded_Label": y_train_encoded[:10]
})

encoding_verification

,Original_Label,Encoded_Label
0,at-risk,0
1,at-risk,0
2,at-risk,0
3,at-risk,0
4,at-risk,0
5,at-risk,0
6,at-risk,0
7,at-risk,0
8,at-risk,0
9,at-risk,0


## 6. Missing-Value Strategy

The exploratory analysis identified missing values in all thirteen predictors. The highest missing percentages were observed for:

- `stress_level`: 12.00%
- `sleep_duration`: 11.01%
- `sleep_quality`: 8.45%
- `calorie_expenditure`: 7.66%
- `water_intake`: 6.30%

Complete-case deletion is not adopted because it could remove a substantial number of observations.

For numerical predictors, median imputation is selected because it is robust to extreme observations.

For categorical predictors, an explicit `Missing` category is used. This preserves the fact that a value was absent rather than automatically replacing it with the most frequent observed category.

The transformations are fitted only using the training partition.

In [38]:
train_missing_summary = pd.DataFrame({
    "Missing_Count": X_train.isna().sum(),
    "Missing_Percentage":
        (X_train.isna().mean() * 100).round(2)
})

train_missing_summary = (
    train_missing_summary
    .sort_values(
        "Missing_Percentage",
        ascending=False
    )
)

train_missing_summary

,Missing_Count,Missing_Percentage
stress_level,66418,12.03
sleep_duration,60702,11.00
sleep_quality,46733,8.47
calorie_expenditure,42122,7.63
water_intake,34865,6.32
physical_activity_level,29253,5.30
smoking_alcohol,22920,4.15
gender,17100,3.10
bmi,11086,2.01
step_count,11121,2.01


## 7. Define Preprocessing Pipelines

Two related preprocessing configurations are defined.

### 7.1 Scaled Preprocessor

This configuration is intended for:

- Logistic Regression
- Support Vector Machine
- MLP Neural Network

Numerical predictors are median-imputed and standardised. Categorical predictors are assigned an explicit missing category and one-hot encoded.

### 7.2 Unscaled Preprocessor

This configuration is intended for:

- Random Forest

Random Forest models do not require standardisation because tree-based splitting is not based on Euclidean distance or gradient scale in the same way as linear, margin-based, and neural-network approaches.

Using separate preprocessors allows each algorithm to receive technically appropriate input while preserving a consistent experimental framework.

In [39]:
numerical_scaled_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

numerical_scaled_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a f

In [40]:
numerical_unscaled_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

numerical_unscaled_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomiss

In [41]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Missing"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

categorical_pipeline




,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('onehot', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'constant'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",'Missing'
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation.

In [42]:
scaled_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_scaled_pipeline,
            numerical_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ],
    remainder="drop"
)

scaled_preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

In [43]:
unscaled_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_unscaled_pipeline,
            numerical_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ],
    remainder="drop"
)

unscaled_preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

## 8. Leakage-Safe Preprocessing Validation

Before model development, the preprocessing design is tested on a small stratified subset of the training partition.

This diagnostic step verifies that:

- missing values are handled correctly;
- categorical variables are encoded successfully;
- transformed dimensions are valid;
- unknown-category handling is operational.

The final model notebooks should embed these preprocessors directly inside model pipelines so that transformations are fitted only on the relevant training data.

In [44]:
X_preprocess_sample, _, y_preprocess_sample, _ = train_test_split(
    X_train,
    y_train_encoded,
    train_size=10000,
    random_state=RANDOM_STATE,
    stratify=y_train_encoded
)

print(
    "Diagnostic Sample Shape:",
    X_preprocess_sample.shape
)

print(
    "Diagnostic Target Shape:",
    y_preprocess_sample.shape
)

Diagnostic Sample Shape: (10000, 13)
Diagnostic Target Shape: (10000,)


In [45]:
X_sample_scaled = (
    scaled_preprocessor
    .fit_transform(X_preprocess_sample)
)

print(
    "Scaled Transformed Shape:",
    X_sample_scaled.shape
)

print(
    "Scaled Output Type:",
    type(X_sample_scaled)
)

Scaled Transformed Shape: (10000, 31)
Scaled Output Type: <class 'numpy.ndarray'>


In [46]:
X_sample_unscaled = (
    unscaled_preprocessor
    .fit_transform(X_preprocess_sample)
)

print(
    "Unscaled Transformed Shape:",
    X_sample_unscaled.shape
)

print(
    "Unscaled Output Type:",
    type(X_sample_unscaled)
)

Unscaled Transformed Shape: (10000, 31)
Unscaled Output Type: <class 'numpy.ndarray'>


In [47]:
def count_missing_transformed(matrix):
    if hasattr(matrix, "data"):
        return np.isnan(matrix.data).sum()
    return np.isnan(matrix).sum()

scaled_missing = count_missing_transformed(
    X_sample_scaled
)

unscaled_missing = count_missing_transformed(
    X_sample_unscaled
)

print(
    "Missing Values in Scaled Output:",
    scaled_missing
)

print(
    "Missing Values in Unscaled Output:",
    unscaled_missing
)

Missing Values in Scaled Output: 0
Missing Values in Unscaled Output: 0


In [48]:
scaled_feature_names = (
    scaled_preprocessor
    .get_feature_names_out()
)

print(
    "Number of Transformed Features:",
    len(scaled_feature_names)
)

print("\nFirst 25 Transformed Features:")

for feature in scaled_feature_names[:25]:
    print(feature)

Number of Transformed Features: 31

First 25 Transformed Features:
num__sleep_duration
num__heart_rate
num__bmi
num__calorie_expenditure
num__step_count
num__exercise_duration
num__water_intake
cat__diet_type_Missing
cat__diet_type_balanced
cat__diet_type_non-veg
cat__diet_type_veg
cat__stress_level_Missing
cat__stress_level_high
cat__stress_level_low
cat__stress_level_medium
cat__sleep_quality_Missing
cat__sleep_quality_average
cat__sleep_quality_good
cat__sleep_quality_poor
cat__physical_activity_level_Missing
cat__physical_activity_level_active
cat__physical_activity_level_moderate
cat__physical_activity_level_sedentary
cat__smoking_alcohol_Missing
cat__smoking_alcohol_no


In [49]:
# Convert safely whether output is sparse or dense
if hasattr(X_sample_scaled, "toarray"):
    scaled_sample_dense = X_sample_scaled.toarray()
else:
    scaled_sample_dense = np.asarray(X_sample_scaled)

n_num = len(numerical_features)

# Numerical transformed columns appear first
scaled_numeric_part = scaled_sample_dense[:, :n_num]

scaling_check = pd.DataFrame({
    "Feature": numerical_features,
    "Approx_Mean": np.mean(
        scaled_numeric_part,
        axis=0
    ).round(4),
    "Approx_Std": np.std(
        scaled_numeric_part,
        axis=0
    ).round(4)
})

scaling_check

,Feature,Approx_Mean,Approx_Std
0,sleep_duration,0.0,1.0
1,heart_rate,-0.0,1.0
2,bmi,-0.0,1.0
3,calorie_expenditure,-0.0,1.0
4,step_count,0.0,1.0
5,exercise_duration,0.0,1.0
6,water_intake,-0.0,1.0


## 9. Class-Imbalance Preparation

The EDA identified a majority-to-minority ratio of approximately 14.89:1. Therefore, the experiments should not rely on raw accuracy alone.

The model-development notebooks will compare imbalance-aware configurations where supported:

- Logistic Regression: `class_weight="balanced"`
- Random Forest: class-weighted configuration
- Support Vector Machine: `class_weight="balanced"`
- MLP Neural Network: alternative imbalance handling through training strategy and evaluation

Model performance will be assessed using:

- Accuracy
- Balanced Accuracy
- Macro Precision
- Macro Recall
- Macro F1-score
- Class-wise Precision
- Class-wise Recall
- Class-wise F1-score
- Confusion Matrix

This is particularly important because a classifier biased toward the majority `at-risk` category may achieve high raw accuracy while performing poorly on the minority `fit` and `unhealthy` categories.

In [50]:
from sklearn.utils.class_weight import compute_class_weight

encoded_classes = np.unique(y_train_encoded)

balanced_weights = compute_class_weight(
    class_weight="balanced",
    classes=encoded_classes,
    y=y_train_encoded
)

class_weight_encoded = {
    int(class_id): float(weight)
    for class_id, weight
    in zip(encoded_classes, balanced_weights)
}

class_weight_original = {
    inverse_target_mapping[class_id]: weight
    for class_id, weight
    in class_weight_encoded.items()
}

print("Encoded Class Weights:")
print(class_weight_encoded)

print("\nOriginal Label Class Weights:")
print(class_weight_original)

Encoded Class Weights:
{0: 0.38819475061298164, 1: 5.779264284069258, 2: 3.985000397005854}

Original Label Class Weights:
{'at-risk': 0.38819475061298164, 'fit': 5.779264284069258, 'unhealthy': 3.985000397005854}


In [51]:
class_weight_table = pd.DataFrame({
    "Class": list(class_weight_original.keys()),
    "Balanced_Weight":
        list(class_weight_original.values())
})

class_weight_table = (
    class_weight_table
    .sort_values(
        "Balanced_Weight",
        ascending=False
    )
    .reset_index(drop=True)
)

class_weight_table

,Class,Balanced_Weight
0,fit,5.779264
1,unhealthy,3.985000
2,at-risk,0.388195


## 10. Outlier Treatment Decision

The IQR analysis identified relatively limited proportions of potential outliers:

- `calorie_expenditure`: 2.014%
- `water_intake`: 1.761%
- `bmi`: 0.791%
- `heart_rate`: 0.414%
- `sleep_duration`: 0.329%
- `exercise_duration`: 0.015%
- `step_count`: 0.000%

These observations are not automatically removed because:

1. the proportions are relatively small;
2. extreme values may represent valid health and lifestyle observations;
3. statistical outliers are not necessarily data errors;
4. removing observations could alter minority-class patterns.

Therefore, the initial experiments retain the observed values. Model-specific sensitivity can later be examined during evaluation.

## 11. Correlation and Feature-Retention Decision

The strongest numerical correlations identified during EDA were:

- `step_count` and `exercise_duration`: 0.44
- `calorie_expenditure` and `step_count`: 0.40
- `calorie_expenditure` and `exercise_duration`: 0.39

These relationships are moderate rather than severe. No numerical predictor is removed solely because of multicollinearity at this stage.

All thirteen predictors are retained for the initial four-model comparison. Feature reduction may later be considered only if supported by model evaluation, feature importance, coefficient analysis, or controlled ablation experiments.

In [52]:
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

preprocessing_metadata = {
    "random_state": RANDOM_STATE,
    "target_column": TARGET_COLUMN,
    "id_column": ID_COLUMN,

    "numerical_features": numerical_features,
    "categorical_features": categorical_features,

    "target_mapping": target_mapping,
    "inverse_target_mapping": {
        str(k): v
        for k, v
        in inverse_target_mapping.items()
    },

    "train_rows": int(len(X_train)),
    "validation_rows": int(len(X_valid)),
    "competition_test_rows": int(len(X_test)),

    "validation_fraction": 0.20,

    "numerical_imputation": "median",
    "categorical_imputation": "constant: Missing",

    "scaled_models": [
        "Logistic Regression",
        "Support Vector Machine",
        "MLP Neural Network"
    ],

    "unscaled_models": [
        "Random Forest"
    ],

    "class_weight_encoded": {
        str(k): v
        for k, v
        in class_weight_encoded.items()
    }
}

METADATA_PATH = os.path.join(
    PROCESSED_DIR,
    "preprocessing_metadata.json"
)

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        preprocessing_metadata,
        file,
        indent=4
    )

print(
    "Metadata saved to:",
    METADATA_PATH
)

Metadata saved to: ../data/processed/preprocessing_metadata.json


In [53]:
split_indices = pd.DataFrame({
    "index": X.index,
    "split": np.where(
        X.index.isin(X_train.index),
        "train",
        "validation"
    )
})

SPLIT_PATH = os.path.join(
    PROCESSED_DIR,
    "split_indices.csv"
)

split_indices.to_csv(
    SPLIT_PATH,
    index=False
)

print(
    "Split indices saved to:",
    SPLIT_PATH
)

print("\nSplit Counts:")
print(
    split_indices["split"]
    .value_counts()
)

Split indices saved to: ../data/processed/split_indices.csv

Split Counts:
split
train         552070
validation    138018
Name: count, dtype: int64


In [54]:
CLASS_WEIGHT_PATH = os.path.join(
    PROCESSED_DIR,
    "class_weights.csv"
)

class_weight_table.to_csv(
    CLASS_WEIGHT_PATH,
    index=False
)

print(
    "Class weights saved to:",
    CLASS_WEIGHT_PATH
)

Class weights saved to: ../data/processed/class_weights.csv


In [55]:
validation_summary = pd.DataFrame({
    "Check": [
        "Training predictors created",
        "Validation predictors created",
        "Competition test predictors created",
        "Target encoded",
        "Stratified split used",
        "Train-validation overlap absent",
        "Numerical features identified",
        "Categorical features identified",
        "Scaled preprocessor tested",
        "Unscaled preprocessor tested",
        "Missing values handled in diagnostic sample",
        "Class weights calculated",
        "Metadata saved",
        "Split indices saved"
    ],

    "Status": [
        X_train is not None,
        X_valid is not None,
        X_test is not None,
        y_train_encoded is not None,
        True,
        X_train.index.intersection(
            X_valid.index
        ).empty,
        len(numerical_features) == 7,
        len(categorical_features) == 6,
        X_sample_scaled is not None,
        X_sample_unscaled is not None,
        scaled_missing == 0
        and unscaled_missing == 0,
        class_weight_encoded is not None,
        os.path.exists(METADATA_PATH),
        os.path.exists(SPLIT_PATH)
    ]
})

validation_summary

,Check,Status
0,Training predictors created,True
1,Validation predictors created,True
2,Competition test predictors created,True
3,Target encoded,True
4,Stratified split used,True
5,Train-validation overlap absent,True
6,Numerical features identified,True
7,Categorical features identified,True
8,Scaled preprocessor tested,True
9,Unscaled preprocessor tested,True


## 12. Preprocessing Conclusion

This notebook established a reproducible and leakage-aware preprocessing framework for student health-risk classification.

The main decisions were:

- the identifier was excluded from predictive modelling;
- the target variable was encoded while preserving a reversible class mapping;
- an 80:20 stratified train-validation split was used because of severe class imbalance;
- numerical missing values were handled using median imputation;
- categorical missing values were represented using an explicit `Missing` category;
- categorical predictors were transformed using one-hot encoding;
- numerical standardisation was prepared for Logistic Regression, SVM, and MLP models;
- a separate unscaled numerical pathway was prepared for Random Forest;
- potential outliers were retained because their proportions were limited and they may represent valid observations;
- all predictors were retained because the EDA did not identify severe numerical multicollinearity;
- balanced class weights were calculated for imbalance-aware experiments;
- fixed split indices were saved to support fair comparison across all four models.

The next notebook develops Logistic Regression as the first interpretable baseline classifier. Both standard and imbalance-aware configurations should be evaluated using accuracy, balanced accuracy, macro-averaged metrics, class-wise performance, and confusion matrices.